# ECG Multi-Task Detection — Rate-Invariant SSL Architecture
## Stage 1: SimCLR Self-Supervised Pretraining → Stage 2: Dual-Head Fine-Tuning → Stage 3: Ground-Truth Cascade Multiplier

**Key insight**: All beats resampled to 200 samples (1-second physiological grid) regardless of source sampling rate.  
A QRS complex at 360Hz and 100Hz both map to the same morphological shape — eliminating the domain gap.


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install wfdb scipy scikit-learn tqdm -q
print("Installs done.")


Installs done.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, copy, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.signal import resample, find_peaks, resample_poly
from sklearn.metrics import (roc_auc_score, precision_recall_curve,
                             confusion_matrix, f1_score)
import wfdb
from tqdm import tqdm
warnings.filterwarnings('ignore')

# ── Paths (only these carry over from the old architecture) ──────
MITBIH_DIR = r'C:\projectapnea\data\raw\mit-bih-arrhythmia-database-1.0.0'
APNEA_DIR  = r'C:\projectapnea\data\raw\apnea-ecg-database-1.0.0'
SLPDB_DIR  = r'C:\projectapnea\data\raw\mit-bih-polysomnographic-database-1.0.0'
PROC_DIR   = r'C:\projectapnea\data\processed_ssl'
CKPT_DIR   = r'C:\projectapnea\checkpoints_ssl'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────
TARGET_SAMPLES = 200    # all beats → 1-second physiological grid
MAX_BEATS      = 60     # max beats stored per minute window (≥60bpm)
EMBED_DIM      = 256    # backbone output dimension
PROJ_DIM       = 128    # SimCLR projection head dimension
SSL_EPOCHS     = 20     # self-supervised pretraining epochs
FT_EPOCHS      = 40     # fine-tuning epochs
FT_PATIENCE    = 10     # early stopping patience
BATCH_ARR      = 128    # arrhythmia beat batch size
BATCH_AP       = 16     # apnea minute batch size (B*MAX_BEATS beats through backbone)
BATCH_SSL      = 1024    # SSL pretraining batch size
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device      : {DEVICE}")
print(f"Beat samples: {TARGET_SAMPLES} (1-second physiological grid)")
print(f"Max beats/min: {MAX_BEATS}")
print(f"Embed dim   : {EMBED_DIM}")


ModuleNotFoundError: No module named 'torch'

In [2]:
# ═══════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS — Beat extraction, R-peak detection, augmentation
# ═══════════════════════════════════════════════════════════════

def extract_beat_1sec(ecg, r_peak, fs, target=TARGET_SAMPLES):
    """
    Extract 1-second window (300ms pre, 700ms post) centred on R-peak.
    Resample to `target` samples → rate-invariant representation.
    A QRS at 360Hz and 100Hz both produce the same shape in this space.
    """
    pre  = int(0.30 * fs)
    post = int(0.70 * fs)
    s, e = r_peak - pre, r_peak + post
    if s < 0 or e > len(ecg):
        return None
    beat = ecg[s:e]
    std  = beat.std()
    if std < 1e-6:          # flat line — skip
        return None
    beat_norm = (beat - beat.mean()) / std
    if len(beat_norm) == target:
        return beat_norm.astype(np.float32)
    return resample(beat_norm, target).astype(np.float32)


def detect_rpeaks(ecg, fs):
    """
    Adaptive R-peak detection. Tries progressively lower amplitude
    thresholds until a physiologically plausible beat count is found
    (30–180 bpm). Falls back to distance-only constraint if needed.
    """
    min_dist  = int(0.35 * fs)          # max ~170 bpm
    n_sec     = len(ecg) / fs
    min_beats = max(1, int(0.5 * n_sec))   # min 30 bpm
    max_beats = int(3.0 * n_sec)           # max 180 bpm

    pos = ecg[ecg > ecg.mean()]
    if len(pos) == 0:
        return np.array([], dtype=int)

    for pct in [80, 70, 60, 50, 40]:
        h = np.percentile(pos, pct)
        peaks, _ = find_peaks(ecg, distance=min_dist, height=h)
        if min_beats <= len(peaks) <= max_beats:
            return peaks

    # Fallback: no height constraint
    peaks, _ = find_peaks(ecg, distance=min_dist)
    return peaks


def build_beat_sequence(ecg_norm, fs, max_beats=MAX_BEATS):
    """
    Detect R-peaks in a normalised ECG window and extract individual
    beats. Returns list of (TARGET_SAMPLES,) float32 arrays.
    """
    peaks = detect_rpeaks(ecg_norm, fs)
    beats = []
    for r in peaks:
        beat = extract_beat_1sec(ecg_norm, r, fs)
        if beat is not None:
            beats.append(beat)
        if len(beats) >= max_beats:
            break
    return beats

def build_beat_sequence_with_rr(ecg_norm, fs, max_beats=MAX_BEATS):
    peaks = detect_rpeaks(ecg_norm, fs)
    beats, rr_intervals = [], []
    prev_r = None
    for r in peaks:
        beat = extract_beat_1sec(ecg_norm, r, fs)
        if beat is None:
            prev_r = r
            continue
        rr_ms = float((r - prev_r) / fs * 1000) if prev_r is not None else 800.0
        beats.append(beat)
        rr_intervals.append(rr_ms)
        prev_r = r
        if len(beats) >= max_beats:
            break
    return beats, rr_intervals


def beats_to_padded(beat_list, max_beats=MAX_BEATS, target=TARGET_SAMPLES):
    """Pack a variable-length list of beats into padded arrays."""
    X = np.zeros((max_beats, target), dtype=np.float32)
    M = np.zeros(max_beats, dtype=bool)
    n = min(len(beat_list), max_beats)
    if n > 0:
        X[:n] = np.array(beat_list[:n])
        M[:n] = True
    return X, M


print("Utility functions defined.")


Utility functions defined.


In [3]:
# ═══════════════════════════════════════════════════════════════
# PREPROCESSING 1/3 — MIT-BIH Arrhythmia Database
# Beat-level extraction using expert .atr annotations (exact R-peaks)
# Original 360Hz → extract 360-sample window → resample to 200
# ═══════════════════════════════════════════════════════════════

ABNORMAL_SYMS = {'V','F','S','A','a','J','j','E','e','Q'}  # AAMI EC57

def process_mitbih():
    records = sorted(set(
        f.split('.')[0] for f in os.listdir(MITBIH_DIR) if f.endswith('.dat')))
    print(f"MIT-BIH: {len(records)} records found")

    all_beats, all_labels, all_recs = [], [], []

    for rec in tqdm(records, desc="MIT-BIH"):
        path = os.path.join(MITBIH_DIR, rec)
        try:
            record = wfdb.rdrecord(path)
            ann    = wfdb.rdann(path, 'atr')
            ecg    = record.p_signal[:, 0]
            fs     = record.fs             # 360 Hz — use ORIGINAL rate

            for r_samp, sym in zip(ann.sample, ann.symbol):
                if sym not in ABNORMAL_SYMS and sym != 'N':
                    continue
                beat = extract_beat_1sec(ecg, r_samp, fs)
                if beat is None:
                    continue
                all_beats.append(beat)
                all_labels.append(1 if sym in ABNORMAL_SYMS else 0)
                all_recs.append(int(rec))
        except Exception as exc:
            print(f"  Skip {rec}: {exc}")

    X   = np.array(all_beats,  dtype=np.float32)   # (N, 200)
    y   = np.array(all_labels, dtype=np.int8)
    rec = np.array(all_recs,   dtype=np.int32)

    np.save(os.path.join(PROC_DIR, 'mitbih_beats.npy'),   X)
    np.save(os.path.join(PROC_DIR, 'mitbih_labels.npy'),  y)
    np.save(os.path.join(PROC_DIR, 'mitbih_rec_ids.npy'), rec)

    print(f"  Saved {len(y):,} beats | Normal: {(y==0).sum():,} | Abnormal: {(y==1).sum():,}")
    print(f"  Imbalance: {(y==0).sum()/(y==1).sum():.1f}:1")
    return X, y, rec


In [4]:
def process_apnea_ecg():
    all_files = os.listdir(APNEA_DIR)
    records   = sorted(set(
        f.split('.')[0] for f in all_files
        if f.endswith('.dat')
        and len(f.split('.')[0]) == 3
        and not f.startswith('x')))
    print(f"Apnea-ECG: {len(records)} records found")

    seqs, rr_seqs, labels, beat_pool = [], [], [], []

    for rec in tqdm(records, desc="Apnea-ECG"):
        path = os.path.join(APNEA_DIR, rec)
        try:
            record = wfdb.rdrecord(path)
            ann    = wfdb.rdann(path, 'apn')
            ecg    = record.p_signal[:, 0]
            fs     = record.fs

            for ann_t, lbl in zip(ann.sample, ann.symbol):
                if lbl not in ['A', 'N']:
                    continue
                end = ann_t + int(60 * fs)
                if end > len(ecg):
                    continue

                window = ecg[ann_t:end]
                norm_w = (window - window.mean()) / (window.std() + 1e-8)

                # ── Extract beats AND RR intervals together ────
                peaks    = detect_rpeaks(norm_w, fs)
                beats, rrs = [], []
                prev_r   = None

                for r in peaks:
                    beat = extract_beat_1sec(norm_w, r, fs)
                    if beat is None:
                        prev_r = r
                        continue
                    # RR interval in ms — normalised to typical range
                    rr_ms = float((r - prev_r) / fs * 1000) if prev_r is not None else 800.0
                    rr_ms = np.clip(rr_ms, 300, 2000)   # physiological limits
                    beats.append(beat)
                    rrs.append(rr_ms)
                    prev_r = r
                    if len(beats) >= MAX_BEATS:
                        break

                if len(beats) < 5:
                    continue

                X_pad, M_pad = beats_to_padded(beats)

                # RR intervals padded with 800ms (neutral) for empty slots
                rr_pad = np.full(MAX_BEATS, 800.0, dtype=np.float32)
                rr_pad[:len(rrs)] = rrs

                seqs.append((X_pad, M_pad))
                rr_seqs.append(rr_pad)
                labels.append(1 if lbl == 'A' else 0)
                beat_pool.extend(beats)

        except Exception as exc:
            print(f"  Skip {rec}: {exc}")

    N   = len(seqs)
    Xb  = np.stack([s[0] for s in seqs], axis=0)   # (N, MAX_BEATS, 200)
    Mb  = np.stack([s[1] for s in seqs], axis=0)   # (N, MAX_BEATS)
    RRb = np.stack(rr_seqs,              axis=0)   # (N, MAX_BEATS)
    yb  = np.array(labels, dtype=np.int8)

    np.save(os.path.join(PROC_DIR, 'apnea_seqs.npy'),       Xb)
    np.save(os.path.join(PROC_DIR, 'apnea_masks.npy'),      Mb)
    np.save(os.path.join(PROC_DIR, 'apnea_rr.npy'),         RRb)
    np.save(os.path.join(PROC_DIR, 'apnea_labels.npy'),     yb)
    np.save(os.path.join(PROC_DIR, 'apnea_beat_pool.npy'),
            np.array(beat_pool, dtype=np.float32))

    print(f"  Saved {N:,} minutes | Apnea: {(yb==1).sum():,} | Normal: {(yb==0).sum():,}")
    print(f"  Beat pool: {len(beat_pool):,} beats")
    return Xb, Mb, RRb, yb

In [5]:
def process_slpdb():
    records = sorted(set(
        f.split('.')[0] for f in os.listdir(SLPDB_DIR) if f.endswith('.dat')))
    print(f"SLPDB: {len(records)} records found")

    VALID_APNEA = {'H', 'HA', 'OA', 'X', 'CA', 'CAA'}
    OLD_FS, NEW_FS = 250, 100
    WIN_SAMPS = 60 * NEW_FS

    seqs, rr_seqs, ap_lbl, ar_lbl, beat_pool, pat_ids = [], [], [], [], [], []  # ← pat_ids added

    for rec in tqdm(records, desc="SLPDB"):
        path = os.path.join(SLPDB_DIR, rec)
        try:
            record  = wfdb.rdrecord(path)
            ecg_raw = record.p_signal[:, 0]
            st_ann  = wfdb.rdann(path, 'st')
            ecg_ann = wfdb.rdann(path, 'ecg')
            ecg_r    = resample_poly(ecg_raw, 2, 5)
            st_samp  = np.round(st_ann.sample  * (NEW_FS / OLD_FS)).astype(int)
            ecg_samp = np.round(ecg_ann.sample * (NEW_FS / OLD_FS)).astype(int)
            st_notes = np.array(st_ann.aux_note)
            ecg_syms = np.array(ecg_ann.symbol)
            n_windows = len(ecg_r) // WIN_SAMPS

            for i in range(n_windows):
                win_s  = i * WIN_SAMPS
                win_e  = win_s + WIN_SAMPS
                window = ecg_r[win_s:win_e]
                ecg_m       = (ecg_samp >= win_s) & (ecg_samp < win_e)
                syms_in_win = ecg_syms[ecg_m]
                if len(syms_in_win) == 0:
                    continue
                st_m = (st_samp >= win_s) & (st_samp < win_e)
                ap   = int(any(
                    note and any(c in note.split() for c in VALID_APNEA)
                    for note in st_notes[st_m]))
                ar = int(any(s not in ['N', '+', '~'] for s in syms_in_win))
                norm_w = (window - window.mean()) / (window.std() + 1e-8)
                peaks  = detect_rpeaks(norm_w, NEW_FS)
                beats, rrs = [], []
                prev_r = None
                for r in peaks:
                    beat = extract_beat_1sec(norm_w, r, NEW_FS)
                    if beat is None:
                        prev_r = r
                        continue
                    rr_ms = float((r - prev_r) / NEW_FS * 1000) if prev_r is not None else 800.0
                    rr_ms = np.clip(rr_ms, 300, 2000)
                    beats.append(beat)
                    rrs.append(rr_ms)
                    prev_r = r
                    if len(beats) >= MAX_BEATS:
                        break
                if len(beats) < 5:
                    continue
                X_pad, M_pad = beats_to_padded(beats)
                rr_pad = np.full(MAX_BEATS, 800.0, dtype=np.float32)
                rr_pad[:len(rrs)] = rrs
                seqs.append((X_pad, M_pad))
                rr_seqs.append(rr_pad)
                ap_lbl.append(ap)
                ar_lbl.append(ar)
                beat_pool.extend(beats)
                pat_ids.append(rec)   # ← track record for every kept window

        except Exception as exc:
            print(f"  Skip {rec}: {exc}")

    N   = len(seqs)
    Xb  = np.stack([s[0] for s in seqs], axis=0)
    Mb  = np.stack([s[1] for s in seqs], axis=0)
    RRb = np.stack(rr_seqs, axis=0)
    yap = np.array(ap_lbl, dtype=np.int8)
    yar = np.array(ar_lbl, dtype=np.int8)
    pid = np.array(pat_ids)   # ← string array of record names

    np.save(os.path.join(PROC_DIR, 'slpdb_seqs.npy'),        Xb)
    np.save(os.path.join(PROC_DIR, 'slpdb_masks.npy'),       Mb)
    np.save(os.path.join(PROC_DIR, 'slpdb_rr.npy'),          RRb)
    np.save(os.path.join(PROC_DIR, 'slpdb_ap_labels.npy'),   yap)
    np.save(os.path.join(PROC_DIR, 'slpdb_ar_labels.npy'),   yar)
    np.save(os.path.join(PROC_DIR, 'slpdb_patient_ids.npy'), pid)   # ← new
    np.save(os.path.join(PROC_DIR, 'slpdb_beat_pool.npy'),
            np.array(beat_pool, dtype=np.float32))

    print(f"  Saved {N:,} windows | Apnea: {(yap==1).sum():,} | "
          f"Arrhythmia GT: {(yar==1).sum():,} | Patients: {len(np.unique(pid))}")
    return Xb, Mb, RRb, yap, yar

In [8]:
import os

# Delete corrupted file
pool_path = os.path.join(PROC_DIR, 'ssl_beat_pool.npy')
if os.path.exists(pool_path):
    os.remove(pool_path)
    print("Deleted corrupted ssl_beat_pool.npy")

# Rebuild from the three source pools
mit_pool = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
ap_pool  = np.load(os.path.join(PROC_DIR, 'apnea_beat_pool.npy'))
sl_pool  = np.load(os.path.join(PROC_DIR, 'slpdb_beat_pool.npy'))
ssl_pool = np.concatenate([mit_pool, ap_pool, sl_pool], axis=0)
np.save(pool_path, ssl_pool)
print(f"SSL beat pool rebuilt: {len(ssl_pool):,} beats")
del mit_pool, ap_pool, sl_pool

Deleted corrupted ssl_beat_pool.npy
SSL beat pool rebuilt: 1,388,224 beats


In [6]:
def _exists(*names):
    return all(os.path.exists(os.path.join(PROC_DIR, n)) for n in names)

if not _exists('mitbih_beats.npy', 'mitbih_labels.npy', 'mitbih_rec_ids.npy'):
    print("Processing MIT-BIH...")
    mit_X, mit_y, mit_rec = process_mitbih()
else:
    print("MIT-BIH already processed — loading.")
    mit_X   = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
    mit_y   = np.load(os.path.join(PROC_DIR, 'mitbih_labels.npy'))
    mit_rec = np.load(os.path.join(PROC_DIR, 'mitbih_rec_ids.npy'))
    print(f"  {len(mit_y):,} beats loaded")

if not _exists('apnea_seqs.npy', 'apnea_labels.npy', 'apnea_rr.npy'):
    print("\nProcessing Apnea-ECG...")
    ap_X, ap_M, ap_RR, ap_y = process_apnea_ecg()
else:
    print("\nApnea-ECG already processed — loading.")
    ap_X  = np.load(os.path.join(PROC_DIR, 'apnea_seqs.npy'))
    ap_M  = np.load(os.path.join(PROC_DIR, 'apnea_masks.npy'))
    ap_RR = np.load(os.path.join(PROC_DIR, 'apnea_rr.npy'))
    ap_y  = np.load(os.path.join(PROC_DIR, 'apnea_labels.npy'))
    print(f"  {len(ap_y):,} minutes loaded")

if not _exists('slpdb_seqs.npy', 'slpdb_ap_labels.npy', 'slpdb_rr.npy'):
    print("\nProcessing SLPDB...")
    sl_X, sl_M, sl_RR, sl_yap, sl_yar = process_slpdb()
else:
    print("\nSLPDB already processed — loading.")
    sl_X   = np.load(os.path.join(PROC_DIR, 'slpdb_seqs.npy'))
    sl_M   = np.load(os.path.join(PROC_DIR, 'slpdb_masks.npy'))
    sl_RR  = np.load(os.path.join(PROC_DIR, 'slpdb_rr.npy'))
    sl_yap = np.load(os.path.join(PROC_DIR, 'slpdb_ap_labels.npy'))
    sl_yar = np.load(os.path.join(PROC_DIR, 'slpdb_ar_labels.npy'))
    print(f"  {len(sl_yap):,} minutes loaded")

# Pool all beats for SSL pretraining
ssl_pool_path = os.path.join(PROC_DIR, 'ssl_beat_pool.npy')
if not os.path.exists(ssl_pool_path):
    mit_pool = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
    ap_pool  = np.load(os.path.join(PROC_DIR, 'apnea_beat_pool.npy'))
    sl_pool  = np.load(os.path.join(PROC_DIR, 'slpdb_beat_pool.npy'))
    ssl_pool = np.concatenate([mit_pool, ap_pool, sl_pool], axis=0)
    np.save(ssl_pool_path, ssl_pool)
    print(f"\nSSL beat pool: {len(ssl_pool):,} beats from all 3 datasets")
else:
    ssl_pool = np.load(ssl_pool_path)
    print(f"\nSSL beat pool already built: {len(ssl_pool):,} beats")

print(f"\nData summary:")
print(f"  MIT-BIH beats : {len(mit_y):,}")
print(f"  Apnea minutes : {len(ap_y):,}  (RR shape: {ap_RR.shape})")
print(f"  SLPDB minutes : {len(sl_yap):,}  (RR shape: {sl_RR.shape})")
print(f"  SSL pool      : {len(ssl_pool):,} beats")

MIT-BIH already processed — loading.
  86,101 beats loaded

Apnea-ECG already processed — loading.
  17,004 minutes loaded

SLPDB already processed — loading.
  5,126 minutes loaded

SSL beat pool already built: 1,388,224 beats

Data summary:
  MIT-BIH beats : 86,101
  Apnea minutes : 17,004  (RR shape: (17004, 60))
  SLPDB minutes : 5,126  (RR shape: (5126, 60))
  SSL pool      : 1,388,224 beats


In [7]:
# ═══════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE
# ECGBackbone: (B, 1, 200) → (B, 256) rate-invariant embedding
# ProjectionHead: for SSL only, discarded after pretraining
# ApneaHead: TCN over sequence of beat embeddings → apnea logit
# ArrhythmiaHead: MLP on single beat embedding → arrhythmia logit
# DualHeadModel: backbone + both heads, two forward paths
# ═══════════════════════════════════════════════════════════════

class SEBlock1D(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(ch, max(ch // r, 4)), nn.ReLU(inplace=True),
            nn.Linear(max(ch // r, 4), ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1)


class ResBlock1D(nn.Module):
    def __init__(self, ci, co, stride=1, dropout=0.1):
        super().__init__()
        self.c1   = nn.Conv1d(ci, co, 7, stride=stride, padding=3, bias=False)
        self.bn1  = nn.BatchNorm1d(co)
        self.c2   = nn.Conv1d(co, co, 7, padding=3, bias=False)
        self.bn2  = nn.BatchNorm1d(co)
        self.se   = SEBlock1D(co)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.skip = (nn.Sequential(
            nn.Conv1d(ci, co, 1, stride=stride, bias=False),
            nn.BatchNorm1d(co))
            if (stride != 1 or ci != co) else nn.Identity())
    def forward(self, x):
        out = self.relu(self.bn1(self.c1(x)))
        out = self.drop(self.se(self.bn2(self.c2(out))))
        return self.relu(out + self.skip(x))


class ECGBackbone(nn.Module):
    """
    Rate-invariant 1D ResNet.
    Input : (B, 1, TARGET_SAMPLES=200)
    Output: (B, EMBED_DIM=256)
    Shared by arrhythmia and apnea heads during fine-tuning.
    Gets gradients from both tasks simultaneously.
    """
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.stem  = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        self.l1    = ResBlock1D(32,  64,         dropout=0.1)
        self.l2    = ResBlock1D(64,  128, stride=2, dropout=0.1)
        self.l3    = ResBlock1D(128, 256, stride=2, dropout=0.1)
        self.l4    = ResBlock1D(256, embed_dim, stride=2, dropout=0.1)
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.embed_dim = embed_dim

    def forward(self, x):
        # x: (B, 1, 200)
        x = self.stem(x)
        x = self.l4(self.l3(self.l2(self.l1(x))))
        return self.pool(x).squeeze(-1)   # (B, embed_dim)


class ProjectionHead(nn.Module):
    """SimCLR projection head — used only during SSL, not fine-tuning."""
    def __init__(self, embed_dim=EMBED_DIM, proj_dim=PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim), nn.ReLU(inplace=True),
            nn.Linear(embed_dim, proj_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class TCNBlock(nn.Module):
    """Temporal Convolutional block with dilation."""
    def __init__(self, ci, co, dilation):
        super().__init__()
        pad = dilation   # preserves sequence length for kernel=3
        self.c1   = nn.Conv1d(ci, co, 3, padding=pad, dilation=dilation, bias=False)
        self.c2   = nn.Conv1d(co, co, 3, padding=pad, dilation=dilation, bias=False)
        self.bn1  = nn.BatchNorm1d(co)
        self.bn2  = nn.BatchNorm1d(co)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(0.2)
        self.skip = nn.Conv1d(ci, co, 1, bias=False) if ci != co else nn.Identity()
    def forward(self, x):
        out = self.relu(self.bn1(self.c1(x)))
        out = self.drop(self.relu(self.bn2(self.c2(out))))
        return self.relu(out + self.skip(x))


class ApneaHead(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, hidden=128):
        super().__init__()
        self.rr_proj = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(inplace=True))
        # First TCNBlock always gets embed_dim + 32 — RR is zeroed if missing
        self.tcn = nn.Sequential(
            TCNBlock(embed_dim + 32, hidden, 1),
            TCNBlock(hidden, hidden, 2),
            TCNBlock(hidden, hidden, 4),
            TCNBlock(hidden, hidden, 8))
        self.attn_pool = nn.Linear(hidden, 1)
        self.head = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(hidden, 1))

    def forward(self, x, rr=None, mask=None):
        B, T, _ = x.shape
        if rr is not None:
            rr_emb = self.rr_proj(rr)              # (B, T, 32)
        else:
            rr_emb = torch.zeros(B, T, 32, device=x.device, dtype=x.dtype)
        x = torch.cat([x, rr_emb], -1)            # (B, T, embed_dim+32) always
        x = x.transpose(1, 2)                      # (B, C, T)
        if mask is not None:
            x = x * mask.unsqueeze(1).float()
        out  = self.tcn(x)
        attn = self.attn_pool(out.transpose(1,2)).squeeze(-1)
        if mask is not None:
            attn = attn.masked_fill(~mask, -1e4)
        attn   = torch.softmax(attn, dim=-1)
        pooled = (out * attn.unsqueeze(1)).sum(-1)
        return self.head(pooled).squeeze(-1)


class DualHeadModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone   = backbone
        self.arr_head   = ArrhythmiaHead()
        self.apnea_head = ApneaHead()

    def forward_arrhythmia(self, x):
        return self.arr_head(self.backbone(x))

    def forward_apnea(self, x_seq, mask=None, rr=None):
        B, T, L = x_seq.shape
        embeds  = self.backbone(x_seq.reshape(B*T, 1, L)).reshape(B, T, -1)
        return self.apnea_head(embeds, rr=rr, mask=mask)

class ArrhythmiaHead(nn.Module):
    """MLP on a single beat embedding → arrhythmia logit."""
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)


# class DualHeadModel(nn.Module):
#     """
#     Shared backbone + two task heads.
#     MIT-BIH beats  → forward_arrhythmia() → arrhythmia loss only
#     Apnea minutes  → forward_apnea()      → apnea loss only
#     Backbone gets gradients from BOTH — no masking tricks.
#     """
#     def __init__(self, backbone):
#         super().__init__()
#         self.backbone   = backbone
#         self.arr_head   = ArrhythmiaHead()
#         self.apnea_head = ApneaHead()

#     def forward_arrhythmia(self, x):
#         """x: (B, 1, 200) → (B,) logits"""
#         return self.arr_head(self.backbone(x))

#     def forward_apnea(self, x_seq, mask=None):
#         """
#         x_seq: (B, T, 200) — sequence of beats for one minute
#         mask : (B, T) bool — True where beat slot is real, False = padding
#         → (B,) apnea logits
#         """
#         B, T, L = x_seq.shape
#         # Encode all beats through shared backbone simultaneously
#         embeds = self.backbone(x_seq.reshape(B * T, 1, L)).reshape(B, T, -1)
#         return self.apnea_head(embeds, mask)


# ── Sanity checks ────────────────────────────────────────────────
_bb = ECGBackbone().to(DEVICE)
_dm = DualHeadModel(_bb).to(DEVICE)

# Test without RR
_ap = _dm.forward_apnea(torch.randn(4, MAX_BEATS, TARGET_SAMPLES, device=DEVICE))
assert _ap.shape == (4,)
print(f"Apnea (no RR) : {tuple(_ap.shape)}  ✓")

# Test with RR
_ap2 = _dm.forward_apnea(
    torch.randn(4, MAX_BEATS, TARGET_SAMPLES, device=DEVICE),
    mask=torch.ones(4, MAX_BEATS, dtype=torch.bool, device=DEVICE),
    rr=torch.randn(4, MAX_BEATS, 1, device=DEVICE))
assert _ap2.shape == (4,)
print(f"Apnea (with RR): {tuple(_ap2.shape)}  ✓")

_ar = _dm.forward_arrhythmia(torch.randn(4, 1, TARGET_SAMPLES, device=DEVICE))
assert _ar.shape == (4,)
print(f"Arrhythmia    : {tuple(_ar.shape)}  ✓")

del _bb, _dm, _ap, _ap2, _ar
torch.cuda.empty_cache()
print("Architecture OK.")


Apnea (no RR) : (4,)  ✓
Apnea (with RR): (4,)  ✓
Arrhythmia    : (4,)  ✓
Architecture OK.


In [8]:
# ═══════════════════════════════════════════════════════════════
# PYTORCH DATASETS
# ═══════════════════════════════════════════════════════════════

class BeatSSLDataset(Dataset):
    """Returns raw beats — augmentation happens on GPU in training loop."""
    def __init__(self, beats):
        # Load entire pool into RAM once as a tensor
        self.beats = torch.tensor(beats, dtype=torch.float32)
    def __len__(self): return len(self.beats)
    def __getitem__(self, i):
        return self.beats[i], self.beats[i]   # same beat, augmented later


def gpu_augment(beats, target=TARGET_SAMPLES):
    B, device = beats.shape[0], beats.device
    result    = torch.empty_like(beats)

    # 8 scale buckets — one F.interpolate call per bucket instead of B calls
    buckets = torch.tensor([0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.4], device=device)
    chosen  = buckets[torch.randint(0, len(buckets), (B,), device=device)]

    for scale_val in buckets:
        mask    = (chosen == scale_val)
        if not mask.any():
            continue
        new_len = max(40, int(target * scale_val.item()))
        group   = beats[mask].unsqueeze(1)   # (k, 1, target)

        stretched = F.interpolate(
            group, size=new_len, mode='linear', align_corners=False
        ).squeeze(1)   # (k, new_len)

        if new_len >= target:
            s = np.random.randint(0, new_len - target + 1)
            result[mask] = stretched[:, s:s+target]
        else:
            padded = F.pad(stretched, (0, target-new_len), mode='replicate')
            result[mask] = padded

    # Amplitude jitter + noise + temporal mask (already vectorised)
    result = result * torch.empty(B, 1, device=device).uniform_(0.7, 1.3)
    result = result + torch.randn_like(result) * 0.04
    ml     = torch.randint(0, int(target*0.2)+1, (B,), device=device)
    ms     = torch.randint(0, target,             (B,), device=device)
    pos    = torch.arange(target, device=device).unsqueeze(0)
    result = result.masked_fill(
        (pos >= ms.unsqueeze(1)) & (pos < (ms+ml).unsqueeze(1)), 0.0)
    return result






class ArrhythmiaBeatDS(Dataset):
    """Single beats with arrhythmia labels for fine-tuning."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N, 1, 200)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ApneaSeqDS(Dataset):
    def __init__(self, X, M, RR, y):
        self.X  = torch.tensor(X,  dtype=torch.float32)
        self.M  = torch.tensor(M,  dtype=torch.bool)
        self.RR = torch.tensor(RR, dtype=torch.float32).unsqueeze(-1)  # (N, T, 1)
        self.y  = torch.tensor(y,  dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.M[i], self.RR[i], self.y[i]


print("Datasets defined.")


Datasets defined.


In [9]:
import time
path = os.path.join(PROC_DIR, 'ssl_beat_pool.npy')
t0 = time.time()
_ = np.load(path)
print(f"Load time: {time.time()-t0:.2f}s")
# If this prints > 2s, data is still on Drive, not local disk

Load time: 8.50s


In [44]:
  # ═══════════════════════════════════════════════════════════════
# STAGE 1: SimCLR SELF-SUPERVISED PRETRAINING
# NT-Xent contrastive loss on augmented beat pairs.
# Backbone must produce the same embedding for two views of the
# same beat under different virtual sampling rates → rate invariance.
# No labels used. Pooled beats from MIT-BIH + Apnea-ECG + SLPDB.
# ═══════════════════════════════════════════════════════════════

def barlow_twins_loss(z1, z2, lam=0.005):
    """
    BarlowTwins loss — no negative pairs needed.
    Works at batch size 256, doesn't collapse.
    """
    B, D = z1.shape
    # Normalise along batch dimension
    z1_n = (z1 - z1.mean(0)) / (z1.std(0) + 1e-8)
    z2_n = (z2 - z2.mean(0)) / (z2.std(0) + 1e-8)

    # Cross-correlation matrix
    C = (z1_n.T @ z2_n) / B   # (D, D)

    # On-diagonal: push toward 1 (invariance)
    # Off-diagonal: push toward 0 (redundancy reduction)
    on_diag  = (1 - C.diagonal()).pow(2).sum()
    off_diag = (C.fill_diagonal_(0)).pow(2).sum()
    return on_diag + lam * off_diag

def train_ssl():
    print("=" * 60)
    print("STAGE 1: SSL PRETRAINING — FULLY IN-VRAM")
    print("=" * 60)

    # ── Load FULL beat pool into VRAM ──────────────────────────
    print("Loading beat pool into VRAM...")
    raw = np.load(os.path.join(PROC_DIR, 'ssl_beat_pool.npy'))
    print(f"Full pool: {len(raw):,} beats")

    all_beats = torch.tensor(raw, dtype=torch.float32).to(DEVICE)
    del raw
    N = len(all_beats)
    print(f"Beat pool in VRAM: {all_beats.shape}  "
          f"({all_beats.nbytes/1e6:.1f} MB)")

    # ── Model ──────────────────────────────────────────────────
    backbone = ECGBackbone().to(DEVICE)
    proj     = ProjectionHead().to(DEVICE)
    params   = list(backbone.parameters()) + list(proj.parameters())
    opt      = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
    sched    = torch.optim.lr_scheduler.CosineAnnealingLR(
                   opt, T_max=SSL_EPOCHS)
    scaler   = torch.cuda.amp.GradScaler()

    steps_per_epoch = N // BATCH_SSL
    print(f"Steps/epoch: {steps_per_epoch:,} | Batch: {BATCH_SSL}")
    print(f"Total updates: {steps_per_epoch * SSL_EPOCHS:,}")
    print(f"Random baseline: {np.log(2 * BATCH_SSL):.4f}")
    print(f"\nEp    Loss     LR")
    print("-" * 35)

    for ep in range(1, SSL_EPOCHS + 1):
        backbone.train(); proj.train()
        ep_loss = 0.0

        # Shuffle indices entirely on GPU
        perm = torch.randperm(N, device=DEVICE)

        pbar = tqdm(range(steps_per_epoch),
                    desc=f"Ep {ep:02d}/{SSL_EPOCHS}", leave=False)

        for step in pbar:
            idx   = perm[step * BATCH_SSL : (step + 1) * BATCH_SSL]
            batch = all_beats[idx]   # (B, 200) — already on GPU

            # Two independent augmented views
            v1 = gpu_augment(batch.clone()).unsqueeze(1)   # (B, 1, 200)
            v2 = gpu_augment(batch.clone()).unsqueeze(1)

            with torch.cuda.amp.autocast():
                z1   = proj(backbone(v1))
                z2   = proj(backbone(v2))
                loss = nt_xent_loss(z1, z2)

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()

            ep_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        sched.step()
        avg = ep_loss / steps_per_epoch
        print(f"  {ep:02d}    {avg:.4f}   {opt.param_groups[0]['lr']:.2e}")

        # Save checkpoint every 5 epochs in case session dies
        if ep % 5 == 0:
            torch.save(backbone.state_dict(),
                       os.path.join(CKPT_DIR, f'ssl_backbone_ep{ep}.pth'))
            print(f"  Checkpoint saved at epoch {ep}")

    ckpt = os.path.join(CKPT_DIR, 'ssl_backbone.pth')
    torch.save(backbone.state_dict(), ckpt)
    print(f"\nSSL backbone saved → {ckpt}")
    print("Projection head discarded — not used in fine-tuning.")
    return backbone


backbone = train_ssl()

STAGE 1: SSL PRETRAINING — FULLY IN-VRAM
Loading beat pool into VRAM...
Full pool: 1,388,225 beats
Beat pool in VRAM: torch.Size([1388225, 200])  (1110.6 MB)
Steps/epoch: 1,355 | Batch: 1024
Total updates: 27,100
Random baseline: 7.6246

Ep    Loss     LR
-----------------------------------


  01    1.5041   9.94e-04


  02    0.9194   9.76e-04


  03    0.7997   9.46e-04


  04    0.7377   9.05e-04


...

  16    0.5442   9.55e-05


  17    0.5387   5.45e-05


  18    0.5351   2.45e-05


  19    0.5324   6.16e-06


  20    0.5314   0.00e+00
  Checkpoint saved at epoch 20

SSL backbone saved → /content/checkpoints_ssl/ssl_backbone.pth
Projection head discarded — not used in fine-tuning.


In [13]:
backbone = ECGBackbone().to(DEVICE)
backbone.load_state_dict(
    torch.load(os.path.join(CKPT_DIR, 'ssl_backbone.pth'),
               map_location=DEVICE))
backbone.eval()
print("Epoch 15 backbone loaded — ready for fine-tuning.")

Epoch 15 backbone loaded — ready for fine-tuning.


In [14]:
class ApneaSeqDSLazy(Dataset):
    """
    Reads directly from .npy files on disk — never loads full arrays into RAM.
    Uses np.load with mmap_mode='r' so only accessed rows are paged in.
    """
    def __init__(self, X_paths, M_paths, RR_paths, y_paths):
        # Memory-map all arrays — OS pages in only what's needed
        self.X_arrs  = [np.load(p, mmap_mode='r') for p in X_paths]
        self.M_arrs  = [np.load(p, mmap_mode='r') for p in M_paths]
        self.RR_arrs = [np.load(p, mmap_mode='r') for p in RR_paths]
        self.y_arrs  = [np.load(p, mmap_mode='r') for p in y_paths]

        # Build index map: (array_idx, row_idx) for each sample
        self.index_map = []
        for i, y in enumerate(self.y_arrs):
            for j in range(len(y)):
                self.index_map.append((i, j))

    def __len__(self): return len(self.index_map)

    def __getitem__(self, idx):
        i, j = self.index_map[idx]
        X  = torch.tensor(self.X_arrs[i][j],  dtype=torch.float32)
        M  = torch.tensor(self.M_arrs[i][j],  dtype=torch.bool)
        RR = torch.tensor(self.RR_arrs[i][j], dtype=torch.float32).unsqueeze(-1)
        y  = torch.tensor(float(self.y_arrs[i][j]), dtype=torch.float32)
        return X, M, RR, y

In [12]:
# ═══════════════════════════════════════════════════════════════
# STAGE 2: SIMULTANEOUS DUAL-HEAD FINE-TUNING
# Single training loop, two heads, one backbone.
# MIT-BIH batch  → arrhythmia BCE loss  → backbone + arr_head gradients
# Apnea batch    → apnea BCE loss       → backbone + apnea_head gradients
# Backbone gets gradients from BOTH tasks every iteration.
# ═══════════════════════════════════════════════════════════════

def build_dataloaders():
    # ── Arrhythmia: patient-stratified split ──────────────────
    X_b  = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
    y_b  = np.load(os.path.join(PROC_DIR, 'mitbih_labels.npy')).astype(np.float32)
    recs = np.load(os.path.join(PROC_DIR, 'mitbih_rec_ids.npy'))

    u_recs   = np.unique(recs)
    val_recs = u_recs[::5]
    tr_m     = ~np.isin(recs, val_recs)
    va_m     =  np.isin(recs, val_recs)

    X_ar_tr, y_ar_tr = X_b[tr_m], y_b[tr_m]
    X_ar_va, y_ar_va = X_b[va_m], y_b[va_m]

    n_neg    = (y_ar_tr == 0).sum()
    n_pos    = (y_ar_tr == 1).sum()
    w        = np.where(y_ar_tr == 1, n_neg / n_pos, 1.0)
    arr_samp = WeightedRandomSampler(
        torch.tensor(w, dtype=torch.float32), len(w), replacement=True)

    arr_tr_ld = DataLoader(ArrhythmiaBeatDS(X_ar_tr, y_ar_tr),
                           batch_size=BATCH_ARR, sampler=arr_samp, drop_last=True,
                           num_workers=0, pin_memory=True)
    arr_va_ld = DataLoader(ArrhythmiaBeatDS(X_ar_va, y_ar_va),
                           batch_size=BATCH_ARR, shuffle=False,
                           num_workers=0, pin_memory=True)

    print(f"Arrhythmia | Train: {len(X_ar_tr):,} "
          f"(normal:{int((y_ar_tr==0).sum()):,} "
          f"abnormal:{int((y_ar_tr==1).sum()):,}) "
          f"| Val: {len(X_ar_va):,} [{len(val_recs)} patients]")

    # ── Apnea: Apnea-ECG + SLPDB combined ─────────────────────
    # ── Apnea: lazy loading, no concatenation in RAM ───────────
    X_paths  = [os.path.join(PROC_DIR, 'apnea_seqs.npy'),
                os.path.join(PROC_DIR, 'slpdb_seqs.npy')]
    M_paths  = [os.path.join(PROC_DIR, 'apnea_masks.npy'),
                os.path.join(PROC_DIR, 'slpdb_masks.npy')]
    RR_paths = [os.path.join(PROC_DIR, 'apnea_rr.npy'),
                os.path.join(PROC_DIR, 'slpdb_rr.npy')]
    y_paths  = [os.path.join(PROC_DIR, 'apnea_labels.npy'),
                os.path.join(PROC_DIR, 'slpdb_ap_labels.npy')]

    full_ds  = ApneaSeqDSLazy(X_paths, M_paths, RR_paths, y_paths)
    n_total  = len(full_ds)
    n_tr     = int(n_total * 0.8)
    n_va     = n_total - n_tr

    tr_ds, va_ds = torch.utils.data.random_split(
        full_ds, [n_tr, n_va],
        generator=torch.Generator().manual_seed(42))

    ap_tr_ld = DataLoader(tr_ds, batch_size=BATCH_AP, shuffle=True,
                          drop_last=True, num_workers=0, pin_memory=False)
    ap_va_ld = DataLoader(va_ds, batch_size=BATCH_AP, shuffle=False,
                          num_workers=0, pin_memory=False)

    # Get label counts for logging
    all_y = np.concatenate([np.load(p, mmap_mode='r') for p in y_paths])
    print(f"Apnea      | Train: {n_tr:,} | Val: {n_va:,} "
          f"(apnea:{int((all_y==1).sum()):,} normal:{int((all_y==0).sum()):,})")
    del all_y


    return arr_tr_ld, arr_va_ld, ap_tr_ld, ap_va_ld


def train_finetune(backbone):
    print("\n" + "=" * 60)
    print("STAGE 2: DUAL-HEAD FINE-TUNING")
    print("=" * 60)

    arr_tr_ld, arr_va_ld, ap_tr_ld, ap_va_ld = build_dataloaders()

    model = DualHeadModel(backbone).to(DEVICE)

    # Pretrained backbone gets lower LR — protect learned representations
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(),   'lr': 5e-5,  'weight_decay': 1e-3},
        {'params': model.arr_head.parameters(),   'lr': 3e-4,  'weight_decay': 1e-3},
        {'params': model.apnea_head.parameters(), 'lr': 3e-4,  'weight_decay': 1e-3},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)
    use_amp   = DEVICE.type == 'cuda'
    scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_combined = 0.0
    best_state    = None
    patience_ctr  = 0
    arr_iter      = iter(arr_tr_ld)

    print("\nEp   ap_loss  ar_loss  val_AP   val_AR   patience")
    print("-" * 55)

    for ep in range(1, FT_EPOCHS + 1):
        model.train()
        ep_ap = ep_ar = 0.0
    
        pbar = tqdm(ap_tr_ld, desc=f"Ep {ep:02d}/{FT_EPOCHS}", leave=False)
    
        for ap_batch in pbar:
            xb_ap, mb_ap, rr_ap, yb_ap = [t.to(DEVICE) for t in ap_batch]
        
            try:
                xb_ar, yb_ar = next(arr_iter)
            except StopIteration:
                arr_iter = iter(arr_tr_ld)
                xb_ar, yb_ar = next(arr_iter)
            xb_ar, yb_ar = xb_ar.to(DEVICE), yb_ar.to(DEVICE)
        
            # ── Apnea step ─────────────────────────────────────────
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                ap_logits = model.forward_apnea(xb_ap, mb_ap, rr_ap)
                ap_loss   = F.binary_cross_entropy_with_logits(ap_logits, yb_ap)
            scaler.scale(ap_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        
            # ── Arrhythmia step ────────────────────────────────────
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                ar_logits = model.forward_arrhythmia(xb_ar)
                ar_loss   = F.binary_cross_entropy_with_logits(
                    ar_logits, yb_ar * 0.9 + 0.05)
            scaler.scale(ar_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        
            ep_ap += ap_loss.item()
            ep_ar += ar_loss.item()
            pbar.set_postfix(ap=f"{ap_loss.item():.4f}", ar=f"{ar_loss.item():.4f}")
    
        scheduler.step()
        n_bat = len(ap_tr_ld)
        # ── Validation ────────────────────────────────────────
        model.eval()

        # Validation apnea loop — replace the existing one:
        ap_lgs, ap_ys = [], []
        with torch.no_grad():
            for xb, mb, rr, yb in ap_va_ld:
                xb, mb, rr = xb.to(DEVICE), mb.to(DEVICE), rr.to(DEVICE)
                ap_lgs.append(model.forward_apnea(xb, mb, rr).cpu())
                ap_ys.append(yb)
        ap_probs = torch.sigmoid(torch.cat(ap_lgs)).numpy()
        ap_y     = torch.cat(ap_ys).numpy()
        ap_auc   = roc_auc_score(ap_y, ap_probs) if len(np.unique(ap_y)) > 1 else 0.5

        ar_lgs, ar_ys = [], []
        with torch.no_grad():
            for xb, yb in arr_va_ld:
                xb = xb.to(DEVICE)
                ar_lgs.append(model.forward_arrhythmia(xb).cpu())
                ar_ys.append(yb)
        ar_probs = torch.sigmoid(torch.cat(ar_lgs)).numpy()
        ar_y     = torch.cat(ar_ys).numpy()
        ar_auc   = roc_auc_score(ar_y, ar_probs) if len(np.unique(ar_y)) > 1 else 0.5

        combined = ap_auc + ar_auc
        print(f"  {ep:02d}   {ep_ap/n_bat:.4f}   {ep_ar/n_bat:.4f}   "
              f"{ap_auc:.4f}   {ar_auc:.4f}   {patience_ctr}/{FT_PATIENCE}")

        if combined > best_combined + 1e-4:
            best_combined = combined
            best_state    = copy.deepcopy(model.state_dict())
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= FT_PATIENCE:
                print("  Early stopping.")
                break

    model.load_state_dict(best_state)
    ckpt = os.path.join(CKPT_DIR, 'dual_head_model.pth')
    torch.save(model.state_dict(), ckpt)
    print(f"\nFine-tuned model saved → {ckpt}")
    return model


model = train_finetune(backbone)



STAGE 2: DUAL-HEAD FINE-TUNING
Arrhythmia | Train: 64,007 (normal:55,534 abnormal:8,473) | Val: 22,094 [10 patients]
Apnea      | Train: 17,704 | Val: 4,426 (apnea:8,698 normal:13,432)


KeyboardInterrupt: 

In [15]:
model = DualHeadModel(backbone).to(DEVICE)
model.load_state_dict(
    torch.load(os.path.join(CKPT_DIR, 'dual_head_model.pth'),
               map_location=DEVICE))

<All keys matched successfully>

In [16]:
# ═══════════════════════════════════════════════════════════════
# STAGE 3: EVALUATION + CLINICAL RISK MULTIPLIER
# Arrhythmia: evaluated on held-out MIT-BIH patients (unseen records)
# Apnea: evaluated on held-out 20% of combined Apnea-ECG + SLPDB
# Cascade multiplier: from SLPDB GROUND TRUTH beat annotations —
#   the arrhythmia model never runs on SLPDB at all.
# ═══════════════════════════════════════════════════════════════

def find_threshold(y_true, probs, target_recall=None):
    """Find optimal F1 threshold. If target_recall set, constrain to >= it."""
    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    if target_recall is not None:
        valid = rec[:-1] >= target_recall
        idx = np.argmax(f1s[:-1] * valid) if valid.any() else np.argmax(f1s[:-1])
    else:
        idx = np.argmax(f1s[:-1])
    return float(thr[idx]), float(f1s[:-1][idx])


def evaluate(model):
    print("\n" + "=" * 60)
    print("STAGE 3: EVALUATION")
    print("=" * 60)
    model.eval()

    # ══ 1. ARRHYTHMIA — held-out MIT-BIH patients ════════════
    X_b  = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
    y_b  = np.load(os.path.join(PROC_DIR, 'mitbih_labels.npy')).astype(np.float32)
    recs = np.load(os.path.join(PROC_DIR, 'mitbih_rec_ids.npy'))
    va_m = np.isin(recs, np.unique(recs)[::5])

    ar_ds = ArrhythmiaBeatDS(X_b[va_m], y_b[va_m])
    ar_ld = DataLoader(ar_ds, batch_size=256, shuffle=False)

    ar_probs_all, ar_y_all = [], []
    with torch.no_grad():
        for xb, yb in ar_ld:
            ar_probs_all.append(torch.sigmoid(model.forward_arrhythmia(xb.to(DEVICE))).cpu())
            ar_y_all.append(yb)
    ar_probs = torch.cat(ar_probs_all).numpy()
    ar_y     = torch.cat(ar_y_all).numpy()

    ar_auc       = roc_auc_score(ar_y, ar_probs)
    ar_thr, ar_f1 = find_threshold(ar_y.astype(int), ar_probs, target_recall=0.75)
    ar_preds     = (ar_probs >= ar_thr).astype(int)
    ar_acc       = (ar_preds == ar_y.astype(int)).mean()
    ar_cm        = confusion_matrix(ar_y.astype(int), ar_preds)

    print(f"\n  ─── ARRHYTHMIA (held-out MIT-BIH patients) ───")
    print(f"  AUC: {ar_auc:.4f} | F1: {ar_f1:.4f} | Acc: {ar_acc:.4f} | Threshold: {ar_thr:.3f}")
    print(f"  Confusion matrix (rows=true, cols=pred):")
    print(f"  [[TN={ar_cm[0,0]:5d}  FP={ar_cm[0,1]:5d}]")
    print(f"   [FN={ar_cm[1,0]:5d}  TP={ar_cm[1,1]:5d}]]")

    # ══ 2. APNEA — held-out 20% ══════════════════════════════
    # ══ 2. APNEA — held-out 20% ══════════════════════════════
    full_ds = ApneaSeqDSLazy(
        X_paths  = [os.path.join(PROC_DIR, 'apnea_seqs.npy'),
                    os.path.join(PROC_DIR, 'slpdb_seqs.npy')],
        M_paths  = [os.path.join(PROC_DIR, 'apnea_masks.npy'),
                    os.path.join(PROC_DIR, 'slpdb_masks.npy')],
        RR_paths = [os.path.join(PROC_DIR, 'apnea_rr.npy'),
                    os.path.join(PROC_DIR, 'slpdb_rr.npy')],
        y_paths  = [os.path.join(PROC_DIR, 'apnea_labels.npy'),
                    os.path.join(PROC_DIR, 'slpdb_ap_labels.npy')])

    n_total = len(full_ds)
    n_tr    = int(n_total * 0.8)
    n_va    = n_total - n_tr
    _, va_ds = torch.utils.data.random_split(
        full_ds, [n_tr, n_va],
        generator=torch.Generator().manual_seed(42))  # same seed as training

    ap_ld = DataLoader(va_ds, batch_size=BATCH_AP, shuffle=False, num_workers=0)

    ap_probs_all, ap_y_all = [], []
    with torch.no_grad():
        for xb, mb, rr, yb in ap_ld:
            xb, mb, rr = xb.to(DEVICE), mb.to(DEVICE), rr.to(DEVICE)
            ap_probs_all.append(
                torch.sigmoid(model.forward_apnea(xb, mb, rr)).cpu())
            ap_y_all.append(yb)

    ap_probs = torch.cat(ap_probs_all).numpy()
    ap_y     = torch.cat(ap_y_all).numpy()
    ap_auc        = roc_auc_score(ap_y, ap_probs)
    ap_thr, ap_f1 = find_threshold(ap_y.astype(int), ap_probs)
    ap_preds      = (ap_probs >= ap_thr).astype(int)
    ap_acc        = (ap_preds == ap_y.astype(int)).mean()
    ap_cm         = confusion_matrix(ap_y.astype(int), ap_preds)

    print(f"\n  ─── APNEA (held-out 20%) ───")
    print(f"  AUC: {ap_auc:.4f} | F1: {ap_f1:.4f} | Acc: {ap_acc:.4f} | Threshold: {ap_thr:.3f}")
    print(f"  Confusion matrix (rows=true, cols=pred):")
    print(f"  [[TN={ap_cm[0,0]:5d}  FP={ap_cm[0,1]:5d}]")
    print(f"   [FN={ap_cm[1,0]:5d}  TP={ap_cm[1,1]:5d}]]")

    # ══ 3. CASCADE MULTIPLIER — uses ap_thr from above ═══════

    # ══ 3. CASCADE MULTIPLIER — SLPDB ground truth ════════════
    # Apnea model predicts on full SLPDB.
    # Arrhythmia label comes from SLPDB .ecg beat annotations directly.
    # The arrhythmia model never touches SLPDB — no cross-domain leakage.
    print(f"\n  ─── CLINICAL RISK MULTIPLIER (SLPDB ground truth) ───")

    # ══ 3. CASCADE MULTIPLIER ════════════════════════════════
    sl_ds = ApneaSeqDSLazy(
        X_paths  = [os.path.join(PROC_DIR, 'slpdb_seqs.npy')],
        M_paths  = [os.path.join(PROC_DIR, 'slpdb_masks.npy')],
        RR_paths = [os.path.join(PROC_DIR, 'slpdb_rr.npy')],
        y_paths  = [os.path.join(PROC_DIR, 'slpdb_ap_labels.npy')])
    sl_ld = DataLoader(sl_ds, batch_size=BATCH_AP, shuffle=False, num_workers=0)

    sl_ap_probs_all = []
    with torch.no_grad():
        for xb, mb, rr, yb in sl_ld:
            xb, mb, rr = xb.to(DEVICE), mb.to(DEVICE), rr.to(DEVICE)
            sl_ap_probs_all.append(
                torch.sigmoid(model.forward_apnea(xb, mb, rr)).cpu())
    sl_ap_probs = torch.cat(sl_ap_probs_all).numpy()

    
    sl_ap_preds = (sl_ap_probs >= 0.5)   # ← this line was missing

    y_sl_ar = np.load(os.path.join(PROC_DIR, 'slpdb_ar_labels.npy')).astype(bool)

    r_ap  = y_sl_ar[ sl_ap_preds].mean() if  sl_ap_preds.sum() > 0 else 0.0
    r_no  = y_sl_ar[~sl_ap_preds].mean() if (~sl_ap_preds).sum() > 0 else 0.0
    mult  = r_ap / r_no if r_no > 0 else float('inf')

    print(f"  Windows predicted apnea  : {sl_ap_preds.sum():,} / {len(sl_ap_preds):,}")
    print(f"  Arrhythmia rate | Apnea windows  : {r_ap*100:.2f}%  (GT)")
    print(f"  Arrhythmia rate | Normal windows : {r_no*100:.2f}%  (GT)")
    print(f"  Clinical risk multiplier         : {mult:.2f}x")
    print(f"  Literature reference             : 2–4x")

    # ══ SUMMARY ══════════════════════════════════════════════
    print(f"\n  {'='*50}")
    print(f"  FINAL RESULTS SUMMARY")
    print(f"  {'='*50}")
    print(f"  Arrhythmia AUC     : {ar_auc:.4f}")
    print(f"  Arrhythmia F1      : {ar_f1:.4f}  (threshold={ar_thr:.3f}, recall≥0.75)")
    print(f"  Arrhythmia Accuracy: {ar_acc:.4f}")
    print(f"  ─")
    print(f"  Apnea AUC          : {ap_auc:.4f}")
    print(f"  Apnea F1           : {ap_f1:.4f}  (threshold={ap_thr:.3f})")
    print(f"  Apnea Accuracy     : {ap_acc:.4f}")
    print(f"  ─")
    print(f"  Risk Multiplier    : {mult:.2f}x  (ref: 2–4x, from SLPDB GT labels)")

    return dict(ar_auc=ar_auc, ar_f1=ar_f1, ar_acc=ar_acc,
                ap_auc=ap_auc, ap_f1=ap_f1, ap_acc=ap_acc,
                multiplier=mult)


results = evaluate(model)



STAGE 3: EVALUATION

  ─── ARRHYTHMIA (held-out MIT-BIH patients) ───
  AUC: 0.8795 | F1: 0.5882 | Acc: 0.8754 | Threshold: 0.056
  Confusion matrix (rows=true, cols=pred):
  [[TN=17373  FP= 2099]
   [FN=  655  TP= 1967]]

  ─── APNEA (held-out 20%) ───
  AUC: 0.9617 | F1: 0.8800 | Acc: 0.9062 | Threshold: 0.139
  Confusion matrix (rows=true, cols=pred):
  [[TN= 2490  FP=  172]
   [FN=  243  TP= 1521]]

  ─── CLINICAL RISK MULTIPLIER (SLPDB ground truth) ───
  Windows predicted apnea  : 1,506 / 5,126
  Arrhythmia rate | Apnea windows  : 3.59%  (GT)
  Arrhythmia rate | Normal windows : 4.31%  (GT)
  Clinical risk multiplier         : 0.83x
  Literature reference             : 2–4x

  FINAL RESULTS SUMMARY
  Arrhythmia AUC     : 0.8795
  Arrhythmia F1      : 0.5882  (threshold=0.056, recall≥0.75)
  Arrhythmia Accuracy: 0.8754
  ─
  Apnea AUC          : 0.9617
  Apnea F1           : 0.8800  (threshold=0.139)
  Apnea Accuracy     : 0.9062
  ─
  Risk Multiplier    : 0.83x  (ref: 2–4x, from

In [18]:
def compute_patient_normalised_multiplier(model, proc_dir, device, ap_thr=0.5):
    """
    Per-patient normalised multiplier.
    Instead of: apnea_windows_arr_rate / normal_windows_arr_rate (population)
    Compute:    per-patient (apnea_rate / baseline_rate) then average
    This removes chronic patient-level arrhythmia confounding.
    """
    model.eval()

    X_sl  = np.load(os.path.join(proc_dir, 'slpdb_seqs.npy'))
    M_sl  = np.load(os.path.join(proc_dir, 'slpdb_masks.npy'))
    RR_sl = np.load(os.path.join(proc_dir, 'slpdb_rr.npy'))
    y_sl_ap = np.load(os.path.join(proc_dir, 'slpdb_ap_labels.npy'))

    # We need per-window patient IDs — rebuild from SLPDB record structure
    # Each SLPDB record contributes a contiguous block of windows
    # Replace the entire patient_ids building section with this:

# Rebuild patient IDs by loading the same records in the same order
# as process_slpdb() did — match exactly N windows saved
    records = sorted(set(
        f.split('.')[0] for f in os.listdir(SLPDB_DIR) if f.endswith('.dat')))
    
    VALID_APNEA = {'H', 'HA', 'OA', 'X', 'CA', 'CAA'}
    OLD_FS, NEW_FS = 250, 100
    WIN_SAMPS = 60 * NEW_FS
    N_total = len(y_sl_ap)
    
    patient_ids = []
    for rec in records:
        path = os.path.join(SLPDB_DIR, rec)
        try:
            record   = wfdb.rdrecord(path)
            ecg_raw  = record.p_signal[:, 0]
            ecg_r    = resample_poly(ecg_raw, 2, 5)
            ecg_ann  = wfdb.rdann(path, 'ecg')
            ecg_samp = np.round(ecg_ann.sample * (NEW_FS / OLD_FS)).astype(int)
            ecg_syms = np.array(ecg_ann.symbol)
            n_windows = len(ecg_r) // WIN_SAMPS
    
            for i in range(n_windows):
                win_s = i * WIN_SAMPS
                win_e = win_s + WIN_SAMPS
                ecg_m = (ecg_samp >= win_s) & (ecg_samp < win_e)
                # Mirror exact skip conditions from process_slpdb:
                if ecg_syms[ecg_m].size == 0:
                    continue
                # Beat extraction would have needed ≥5 beats — approximate
                # by checking annotation density (>4 beats in window)
                n_beats = (ecg_m).sum()
                if n_beats < 5:
                    continue
                patient_ids.append(rec)
                if len(patient_ids) >= N_total:
                    break
        except:
            continue
        if len(patient_ids) >= N_total:
            break

    patient_ids = np.array(patient_ids)
    print(f"Patient IDs rebuilt: {len(patient_ids)} windows across "
          f"{len(np.unique(patient_ids))} patients")
    assert len(patient_ids) == N_total, \
        f"Mismatch: {len(patient_ids)} IDs vs {N_total} windows — " \
        f"save patient_ids during preprocessing instead"

    # ── Get arrhythmia scores from model ──────────────────────
    print("Scoring SLPDB windows through arrhythmia model...")
    arr_scores = np.zeros(len(y_sl_ap), dtype=np.float32)
    bs = 32
    for i in tqdm(range(0, len(y_sl_ap), bs)):
        xb = torch.tensor(X_sl[i:i+bs], dtype=torch.float32).to(device)
        mb = torch.tensor(M_sl[i:i+bs], dtype=torch.bool).to(device)
        B_, T_, L_ = xb.shape
        with torch.no_grad():
            embeds     = model.backbone(
                xb.reshape(B_*T_, 1, L_)).reshape(B_, T_, -1)
            beat_probs = torch.sigmoid(
                model.arr_head(embeds.reshape(B_*T_, -1))
            ).reshape(B_, T_).cpu().numpy()
        mask_np = mb.cpu().numpy()
        for j in range(B_):
            real = beat_probs[j][mask_np[j]]
            if len(real) > 0:
                arr_scores[i+j] = real.mean()

    # ── Apnea predictions ─────────────────────────────────────
    sl_ds = ApneaSeqDSLazy(
        X_paths  = [os.path.join(proc_dir, 'slpdb_seqs.npy')],
        M_paths  = [os.path.join(proc_dir, 'slpdb_masks.npy')],
        RR_paths = [os.path.join(proc_dir, 'slpdb_rr.npy')],
        y_paths  = [os.path.join(proc_dir, 'slpdb_ap_labels.npy')])
    sl_ld = DataLoader(sl_ds, batch_size=BATCH_AP, shuffle=False, num_workers=0)

    ap_probs_all = []
    with torch.no_grad():
        for xb, mb, rr, yb in sl_ld:
            xb, mb, rr = xb.to(device), mb.to(device), rr.to(device)
            ap_probs_all.append(
                torch.sigmoid(model.forward_apnea(xb, mb, rr)).cpu())
    ap_probs = torch.cat(ap_probs_all).numpy()
    ap_preds = (ap_probs >= ap_thr)

    # ── Per-patient normalised multiplier ─────────────────────
    print(f"\n  ─── PER-PATIENT NORMALISED MULTIPLIER ───")
    print(f"  {'Patient':>10}  {'AP_wins':>8}  {'NO_wins':>8}  "
          f"{'r_ap':>7}  {'r_no':>7}  {'ratio':>7}")

    ratios = []
    # Only include patients with balanced apnea burden (20-80%)
    for pat in np.unique(patient_ids):
        pat_m   = patient_ids == pat
        ap_rate = ap_preds[pat_m].mean()
        if not (0.20 <= ap_rate <= 0.80):   # skip extreme cases
            print(f"  {pat} skipped — apnea burden {ap_rate*100:.0f}%")
            continue
    # rest of per-patient calculation
        ap_m     = pat_m &  ap_preds
        no_m     = pat_m & ~ap_preds
        if ap_m.sum() < 3 or no_m.sum() < 3:
            continue
        r_ap = arr_scores[ap_m].mean()
        r_no = arr_scores[no_m].mean()
        if r_no < 1e-6:
            continue
        ratio = r_ap / r_no
        ratios.append(ratio)
        print(f"  {pat:>10}  {ap_m.sum():>8,}  {no_m.sum():>8,}  "
              f"{r_ap:>7.3f}  {r_no:>7.3f}  {ratio:>7.2f}x")

    if ratios:
        median_mult = float(np.median(ratios))
        mean_mult   = float(np.mean(ratios))
        print(f"\n  Median per-patient multiplier: {median_mult:.2f}x")
        print(f"  Mean   per-patient multiplier: {mean_mult:.2f}x")
        print(f"  Literature reference         : 1–4x")
        return median_mult
    return 0.0


patient_mult = compute_patient_normalised_multiplier(model, PROC_DIR, DEVICE)

Patient IDs rebuilt: 5126 windows across 18 patients
Scoring SLPDB windows through arrhythmia model...


100%|████████████████████████████████████████████████████████████████████████████████| 161/161 [00:29<00:00,  5.42it/s]



  ─── PER-PATIENT NORMALISED MULTIPLIER ───
     Patient   AP_wins   NO_wins     r_ap     r_no    ratio
  slp01a skipped — apnea burden 0%
  slp01b skipped — apnea burden 8%
  slp02a skipped — apnea burden 13%
  slp02b skipped — apnea burden 6%
  slp03 skipped — apnea burden 20%
       slp04       128       232    0.365    0.412     0.89x
       slp14        94       266    0.301    0.107     2.82x
       slp16       129       231    0.094    0.094     1.00x
  slp32 skipped — apnea burden 9%
  slp37 skipped — apnea burden 80%
  slp41 skipped — apnea burden 1%
  slp45 skipped — apnea burden 0%
       slp48       152       228    0.580    0.719     0.81x
       slp59        95       145    0.061    0.050     1.22x
       slp60       171       181    0.078    0.055     1.41x
       slp61       131       239    0.081    0.055     1.48x
       slp66       125        95    0.050    0.050     1.00x
      slp67x        50        27    0.050    0.050     0.99x

  Median per-patient multiplier:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTIONAL: Resume from saved checkpoints (skip retraining)
# ═══════════════════════════════════════════════════════════════

# def load_model_from_checkpoint():
#     """Load fine-tuned model directly without retraining."""
#     backbone  = ECGBackbone().to(DEVICE)
#     ssl_ckpt  = os.path.join(CKPT_DIR, 'ssl_backbone.pth')
#     if os.path.exists(ssl_ckpt):
#         backbone.load_state_dict(
#             torch.load(ssl_ckpt, map_location=DEVICE))
#         print(f"SSL backbone loaded from {ssl_ckpt}")

#     model = DualHeadModel(backbone).to(DEVICE)
#     ft_ckpt = os.path.join(CKPT_DIR, 'dual_head_model.pth')
#     if os.path.exists(ft_ckpt):
#         model.load_state_dict(
#             torch.load(ft_ckpt, map_location=DEVICE))
#         print(f"Fine-tuned model loaded from {ft_ckpt}")
#     else:
#         print("No fine-tuned checkpoint found. Run training first.")
#     return model

# Uncomment to skip retraining and evaluate saved model:
model = load_model_from_checkpoint()
results = evaluate(model)


In [19]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE EVALUATION — Full metrics suite
# ═══════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, f1_score, classification_report,
    average_precision_score, matthews_corrcoef,
    cohen_kappa_score, brier_score_loss,
    balanced_accuracy_score)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ────────────────────────────────────────────
C_BG    = '#0d1117'
C_PANEL = '#161b22'
C_GRID  = '#21262d'
C_TEXT  = '#e6edf3'
C_DIM   = '#8b949e'
C_AP    = '#58a6ff'   # apnea — blue
C_AR    = '#f78166'   # arrhythmia — coral
C_GREEN = '#3fb950'
C_GOLD  = '#d29922'

plt.rcParams.update({
    'figure.facecolor':  C_BG,
    'axes.facecolor':    C_PANEL,
    'axes.edgecolor':    C_GRID,
    'axes.labelcolor':   C_TEXT,
    'axes.titlecolor':   C_TEXT,
    'xtick.color':       C_DIM,
    'ytick.color':       C_DIM,
    'grid.color':        C_GRID,
    'text.color':        C_TEXT,
    'legend.facecolor':  C_PANEL,
    'legend.edgecolor':  C_GRID,
    'font.family':       'monospace',
    'font.size':         9,
})


def bootstrap_ci(y_true, y_score, metric_fn, n=1000, ci=0.95):
    """Bootstrap confidence interval for any scalar metric."""
    vals = []
    rng  = np.random.default_rng(42)
    for _ in range(n):
        idx  = rng.integers(0, len(y_true), len(y_true))
        try:
            vals.append(metric_fn(y_true[idx], y_score[idx]))
        except Exception:
            pass
    lo = np.percentile(vals, (1 - ci) / 2 * 100)
    hi = np.percentile(vals, (1 + ci) / 2 * 100)
    return float(np.mean(vals)), lo, hi


def compute_all_metrics(y_true, y_prob, task_name, threshold=None):
    """Return a dict of every metric we care about."""
    y_true = y_true.astype(int)

    if threshold is None:
        prec, rec, thr = precision_recall_curve(y_true, y_prob)
        f1s = 2 * prec * rec / (prec + rec + 1e-8)
        threshold = float(thr[np.argmax(f1s[:-1])])

    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity  = tp / (tp + fn + 1e-8)
    specificity  = tn / (tn + fp + 1e-8)
    ppv          = tp / (tp + fp + 1e-8)
    npv          = tn / (tn + fn + 1e-8)
    f1           = 2 * ppv * sensitivity / (ppv + sensitivity + 1e-8)
    auc          = roc_auc_score(y_true, y_prob)
    ap           = average_precision_score(y_true, y_prob)
    mcc          = matthews_corrcoef(y_true, y_pred)
    kappa        = cohen_kappa_score(y_true, y_pred)
    brier        = brier_score_loss(y_true, y_prob)
    bal_acc      = balanced_accuracy_score(y_true, y_pred)
    acc          = (tp + tn) / (tp + tn + fp + fn)
    lr_pos       = sensitivity / (1 - specificity + 1e-8)
    lr_neg       = (1 - sensitivity) / (specificity + 1e-8)

    # Bootstrap CI for AUC
    auc_mean, auc_lo, auc_hi = bootstrap_ci(
        y_true, y_prob, roc_auc_score, n=500)

    return dict(
        task=task_name, threshold=threshold,
        auc=auc, auc_lo=auc_lo, auc_hi=auc_hi,
        ap=ap, f1=f1, mcc=mcc, kappa=kappa,
        sensitivity=sensitivity, specificity=specificity,
        ppv=ppv, npv=npv, brier=brier,
        bal_acc=bal_acc, acc=acc,
        lr_pos=lr_pos, lr_neg=lr_neg,
        tp=tp, tn=tn, fp=fp, fn=fn,
        y_true=y_true, y_prob=y_prob, y_pred=y_pred)


def plot_full_report(m_ap, m_ar, save_path='evaluation_report.png'):
    fig = plt.figure(figsize=(20, 24), facecolor=C_BG)
    gs  = gridspec.GridSpec(4, 4, figure=fig,
                            hspace=0.45, wspace=0.35,
                            left=0.06, right=0.97,
                            top=0.93, bottom=0.04)

    # ── Title ─────────────────────────────────────────────────
    fig.text(0.5, 0.965, 'ECG Multi-Task Detection — Evaluation Report',
             ha='center', va='center', fontsize=16, fontweight='bold',
             color=C_TEXT, fontfamily='monospace')
    fig.text(0.5, 0.950,
             'Rate-Invariant SSL Backbone  ·  Dual-Head Fine-Tuning  ·  '
             'MIT-BIH + Apnea-ECG + SLPDB',
             ha='center', va='center', fontsize=9, color=C_DIM)

    # ════════════════════════════════════════════════════════
    # ROW 0 — ROC curves (left) + PR curves (right)
    # ════════════════════════════════════════════════════════
    ax_roc = fig.add_subplot(gs[0, :2])
    ax_pr  = fig.add_subplot(gs[0, 2:])

    for m, color, label in [(m_ap, C_AP, 'Apnea'),
                             (m_ar, C_AR, 'Arrhythmia')]:
        fpr, tpr, _ = roc_curve(m['y_true'], m['y_prob'])
        ax_roc.plot(fpr, tpr, color=color, lw=2,
                    label=f"{label}  AUC={m['auc']:.4f} "
                          f"[{m['auc_lo']:.3f}–{m['auc_hi']:.3f}]")
        ax_roc.fill_between(fpr, tpr, alpha=0.08, color=color)

        # Mark operating point
        thr_idx = np.argmin(np.abs(
            np.array([m['threshold']] * len(fpr)) -
            np.interp(fpr, fpr, fpr)))
        op_fpr = 1 - m['specificity']
        op_tpr = m['sensitivity']
        ax_roc.scatter([op_fpr], [op_tpr], color=color,
                       s=80, zorder=5, marker='D')

        prec, rec, _ = precision_recall_curve(m['y_true'], m['y_prob'])
        ax_pr.plot(rec, prec, color=color, lw=2,
                   label=f"{label}  AP={m['ap']:.4f}")
        ax_pr.fill_between(rec, prec, alpha=0.08, color=color)

    ax_roc.plot([0,1],[0,1], '--', color=C_GRID, lw=1.2, label='Random')
    ax_roc.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
               title='ROC Curves  (◆ = operating point)')
    ax_roc.legend(fontsize=8)
    ax_roc.grid(True, alpha=0.3)

    ax_pr.set(xlabel='Recall', ylabel='Precision',
              title='Precision–Recall Curves')
    ax_pr.legend(fontsize=8)
    ax_pr.grid(True, alpha=0.3)

    # ════════════════════════════════════════════════════════
    # ROW 1 — Confusion matrices
    # ════════════════════════════════════════════════════════
    for col, (m, color, label) in enumerate(
            [(m_ap, C_AP, 'Apnea'), (m_ar, C_AR, 'Arrhythmia')]):
        ax = fig.add_subplot(gs[1, col*2:(col+1)*2])
        cm = np.array([[m['tn'], m['fp']],
                       [m['fn'], m['tp']]])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1,
                       aspect='auto')

        labels = [['TN', 'FP'], ['FN', 'TP']]
        row_lbl = ['Actual Neg', 'Actual Pos']
        col_lbl = ['Pred Neg', 'Pred Pos']

        for i in range(2):
            for j in range(2):
                pct  = cm_norm[i, j] * 100
                cnt  = cm[i, j]
                bg   = cm_norm[i, j]
                tcol = C_BG if bg > 0.5 else C_TEXT
                ax.text(j, i,
                        f"{labels[i][j]}\n{cnt:,}\n({pct:.1f}%)",
                        ha='center', va='center',
                        fontsize=9, color=tcol, fontweight='bold')

        ax.set_xticks([0,1]); ax.set_xticklabels(col_lbl)
        ax.set_yticks([0,1]); ax.set_yticklabels(row_lbl)
        ax.set_title(f'{label} Confusion Matrix\n'
                     f'Threshold = {m["threshold"]:.3f}',
                     color=color)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # ════════════════════════════════════════════════════════
    # ROW 2 — Probability distributions + calibration
    # ════════════════════════════════════════════════════════
    for col, (m, color, label) in enumerate(
            [(m_ap, C_AP, 'Apnea'), (m_ar, C_AR, 'Arrhythmia')]):

        # Probability distribution
        ax_dist = fig.add_subplot(gs[2, col*2])
        pos_mask = m['y_true'] == 1
        neg_mask = m['y_true'] == 0
        bins = np.linspace(0, 1, 40)
        ax_dist.hist(m['y_prob'][neg_mask], bins=bins,
                     alpha=0.6, color=C_DIM, label='Negative', density=True)
        ax_dist.hist(m['y_prob'][pos_mask], bins=bins,
                     alpha=0.6, color=color, label='Positive', density=True)
        ax_dist.axvline(m['threshold'], color=C_GOLD, lw=1.5,
                        linestyle='--', label=f"Threshold={m['threshold']:.3f}")
        ax_dist.set(xlabel='Predicted Probability', ylabel='Density',
                    title=f'{label} Score Distribution')
        ax_dist.legend(fontsize=7)
        ax_dist.grid(True, alpha=0.3)

        # Calibration curve
        ax_cal = fig.add_subplot(gs[2, col*2+1])
        n_bins  = 10
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_means, frac_pos = [], []
        for b in range(n_bins):
            lo_, hi_ = bin_edges[b], bin_edges[b+1]
            mask = (m['y_prob'] >= lo_) & (m['y_prob'] < hi_)
            if mask.sum() > 0:
                bin_means.append(m['y_prob'][mask].mean())
                frac_pos.append(m['y_true'][mask].mean())
        ax_cal.plot([0,1],[0,1], '--', color=C_GRID, lw=1.2,
                    label='Perfect calibration')
        ax_cal.plot(bin_means, frac_pos, 'o-', color=color,
                    lw=2, ms=5, label='Model')
        ax_cal.fill_between(bin_means, frac_pos,
                            [bm for bm in bin_means],
                            alpha=0.15, color=color)
        ax_cal.set(xlabel='Mean Predicted Probability',
                   ylabel='Fraction of Positives',
                   title=f'{label} Calibration Curve\n'
                         f'Brier Score = {m["brier"]:.4f}')
        ax_cal.legend(fontsize=7)
        ax_cal.grid(True, alpha=0.3)

    # ════════════════════════════════════════════════════════
    # ROW 3 — Metrics table
    # ════════════════════════════════════════════════════════
    ax_tbl = fig.add_subplot(gs[3, :])
    ax_tbl.axis('off')

    metrics_display = [
        ('AUC-ROC',         f"{m_ap['auc']:.4f} [{m_ap['auc_lo']:.3f}–{m_ap['auc_hi']:.3f}]",
                            f"{m_ar['auc']:.4f} [{m_ar['auc_lo']:.3f}–{m_ar['auc_hi']:.3f}]"),
        ('Avg Precision',   f"{m_ap['ap']:.4f}",    f"{m_ar['ap']:.4f}"),
        ('F1 Score',        f"{m_ap['f1']:.4f}",    f"{m_ar['f1']:.4f}"),
        ('Sensitivity',     f"{m_ap['sensitivity']:.4f}",  f"{m_ar['sensitivity']:.4f}"),
        ('Specificity',     f"{m_ap['specificity']:.4f}",  f"{m_ar['specificity']:.4f}"),
        ('PPV (Precision)', f"{m_ap['ppv']:.4f}",   f"{m_ar['ppv']:.4f}"),
        ('NPV',             f"{m_ap['npv']:.4f}",   f"{m_ar['npv']:.4f}"),
        ('Balanced Acc',    f"{m_ap['bal_acc']:.4f}", f"{m_ar['bal_acc']:.4f}"),
        ('MCC',             f"{m_ap['mcc']:.4f}",   f"{m_ar['mcc']:.4f}"),
        ("Cohen's Kappa",   f"{m_ap['kappa']:.4f}", f"{m_ar['kappa']:.4f}"),
        ('Brier Score',     f"{m_ap['brier']:.4f}", f"{m_ar['brier']:.4f}"),
        ('LR+',             f"{m_ap['lr_pos']:.2f}x", f"{m_ar['lr_pos']:.2f}x"),
        ('LR-',             f"{m_ap['lr_neg']:.4f}", f"{m_ar['lr_neg']:.4f}"),
        ('Threshold',       f"{m_ap['threshold']:.4f}", f"{m_ar['threshold']:.4f}"),
    ]

    col_labels  = ['Metric', 'Apnea', 'Arrhythmia']
    col_colors  = [C_GRID,   C_AP,    C_AR]
    cell_colors = []
    cell_text   = []

    for row in metrics_display:
        cell_text.append(list(row))
        cell_colors.append([C_PANEL, C_PANEL, C_PANEL])

    tbl = ax_tbl.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1])

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_facecolor(C_PANEL if r > 0 else C_GRID)
        cell.set_edgecolor(C_GRID)
        cell.set_text_props(
            color=C_TEXT if r > 0 else C_TEXT,
            fontweight='bold' if r == 0 else 'normal',
            fontfamily='monospace')
        if r > 0 and c == 1:
            cell.set_text_props(color=C_AP)
        if r > 0 and c == 2:
            cell.set_text_props(color=C_AR)

    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=C_BG, edgecolor='none')
    print(f"Report saved → {save_path}")
    return fig


def run_full_evaluation(model):
    model.eval()
    print("Running full evaluation...")

    # ── Arrhythmia ────────────────────────────────────────────
    X_b  = np.load(os.path.join(PROC_DIR, 'mitbih_beats.npy'))
    y_b  = np.load(os.path.join(PROC_DIR, 'mitbih_labels.npy')).astype(np.float32)
    recs = np.load(os.path.join(PROC_DIR, 'mitbih_rec_ids.npy'))
    va_m = np.isin(recs, np.unique(recs)[::5])

    ar_ds = ArrhythmiaBeatDS(X_b[va_m], y_b[va_m])
    ar_ld = DataLoader(ar_ds, batch_size=512, shuffle=False, num_workers=0)

    ar_probs_all, ar_y_all = [], []
    with torch.no_grad():
        for xb, yb in tqdm(ar_ld, desc='Arrhythmia eval', leave=False):
            ar_probs_all.append(
                torch.sigmoid(model.forward_arrhythmia(xb.to(DEVICE))).cpu())
            ar_y_all.append(yb)
    ar_probs = torch.cat(ar_probs_all).numpy()
    ar_y     = torch.cat(ar_y_all).numpy()

    # ── Apnea ─────────────────────────────────────────────────
    full_ds = ApneaSeqDSLazy(
        X_paths  = [os.path.join(PROC_DIR, 'apnea_seqs.npy'),
                    os.path.join(PROC_DIR, 'slpdb_seqs.npy')],
        M_paths  = [os.path.join(PROC_DIR, 'apnea_masks.npy'),
                    os.path.join(PROC_DIR, 'slpdb_masks.npy')],
        RR_paths = [os.path.join(PROC_DIR, 'apnea_rr.npy'),
                    os.path.join(PROC_DIR, 'slpdb_rr.npy')],
        y_paths  = [os.path.join(PROC_DIR, 'apnea_labels.npy'),
                    os.path.join(PROC_DIR, 'slpdb_ap_labels.npy')])
    n_total = len(full_ds)
    n_tr    = int(n_total * 0.8)
    _, va_ds = torch.utils.data.random_split(
        full_ds, [n_tr, n_total - n_tr],
        generator=torch.Generator().manual_seed(42))
    ap_ld = DataLoader(va_ds, batch_size=BATCH_AP, shuffle=False, num_workers=0)

    ap_probs_all, ap_y_all = [], []
    with torch.no_grad():
        for xb, mb, rr, yb in tqdm(ap_ld, desc='Apnea eval', leave=False):
            xb, mb, rr = xb.to(DEVICE), mb.to(DEVICE), rr.to(DEVICE)
            ap_probs_all.append(
                torch.sigmoid(model.forward_apnea(xb, mb, rr)).cpu())
            ap_y_all.append(yb)
    ap_probs = torch.cat(ap_probs_all).numpy()
    ap_y     = torch.cat(ap_y_all).numpy()

    # ── Compute metrics ───────────────────────────────────────
    m_ap = compute_all_metrics(ap_y, ap_probs, 'Apnea')
    m_ar = compute_all_metrics(ar_y, ar_probs, 'Arrhythmia',
                               threshold=None)  # finds best F1 threshold

    # ── Print summary ─────────────────────────────────────────
    print(f"\n{'═'*58}")
    print(f"  {'METRIC':<22} {'APNEA':>16} {'ARRHYTHMIA':>16}")
    print(f"{'─'*58}")
    rows = [
        ('AUC-ROC (95% CI)',
         f"{m_ap['auc']:.4f} [{m_ap['auc_lo']:.3f}–{m_ap['auc_hi']:.3f}]",
         f"{m_ar['auc']:.4f} [{m_ar['auc_lo']:.3f}–{m_ar['auc_hi']:.3f}]"),
        ('Avg Precision (AP)', f"{m_ap['ap']:.4f}", f"{m_ar['ap']:.4f}"),
        ('F1 Score',           f"{m_ap['f1']:.4f}", f"{m_ar['f1']:.4f}"),
        ('Sensitivity/Recall', f"{m_ap['sensitivity']:.4f}", f"{m_ar['sensitivity']:.4f}"),
        ('Specificity',        f"{m_ap['specificity']:.4f}", f"{m_ar['specificity']:.4f}"),
        ('PPV (Precision)',    f"{m_ap['ppv']:.4f}", f"{m_ar['ppv']:.4f}"),
        ('NPV',                f"{m_ap['npv']:.4f}", f"{m_ar['npv']:.4f}"),
        ('Balanced Accuracy',  f"{m_ap['bal_acc']:.4f}", f"{m_ar['bal_acc']:.4f}"),
        ('MCC',                f"{m_ap['mcc']:.4f}", f"{m_ar['mcc']:.4f}"),
        ("Cohen's Kappa",      f"{m_ap['kappa']:.4f}", f"{m_ar['kappa']:.4f}"),
        ('Brier Score ↓',      f"{m_ap['brier']:.4f}", f"{m_ar['brier']:.4f}"),
        ('LR+ (>1 = good)',    f"{m_ap['lr_pos']:.2f}x", f"{m_ar['lr_pos']:.2f}x"),
        ('LR- (<1 = good)',    f"{m_ap['lr_neg']:.4f}", f"{m_ar['lr_neg']:.4f}"),
        ('Threshold',          f"{m_ap['threshold']:.4f}", f"{m_ar['threshold']:.4f}"),
        ('TP / FP / TN / FN',
         f"{m_ap['tp']}/{m_ap['fp']}/{m_ap['tn']}/{m_ap['fn']}",
         f"{m_ar['tp']}/{m_ar['fp']}/{m_ar['tn']}/{m_ar['fn']}"),
    ]
    for name, v_ap, v_ar in rows:
        print(f"  {name:<22} {v_ap:>16} {v_ar:>16}")
    print(f"{'═'*58}")

    # ── Plot ──────────────────────────────────────────────────
    save_path = os.path.join(CKPT_DIR, 'evaluation_report.png')
    plot_full_report(m_ap, m_ar, save_path=save_path)

    return m_ap, m_ar


m_ap, m_ar = run_full_evaluation(model)

Running full evaluation...



══════════════════════════════════════════════════════════
  METRIC                            APNEA       ARRHYTHMIA
──────────────────────────────────────────────────────────
  AUC-ROC (95% CI)       0.9617 [0.956–0.967] 0.8795 [0.871–0.888]
  Avg Precision (AP)               0.9476           0.6412
  F1 Score                         0.8800           0.6825
  Sensitivity/Recall               0.8622           0.6404
  Specificity                      0.9354           0.9682
  PPV (Precision)                  0.8984           0.7306
  NPV                              0.9111           0.9524
  Balanced Accuracy                0.8988           0.8043
  MCC                              0.8035           0.6447
  Cohen's Kappa                    0.8031           0.6429
  Brier Score ↓                    0.0904           0.0597
  LR+ (>1 = good)                  13.34x           20.14x
  LR- (<1 = good)                  0.1473           0.3715
  Threshold                        0.1395      